<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-07-neural-networks/07_MLP_Digit_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup & Load MNIST Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import fetch_openml
import wandb

# Inisialisasi W&B
run = wandb.init(project="mnist-neural-networks", name="mlp-baseline")

# Load MNIST Dataset (Angka 0-9)
print("Loading MNIST data... (Mohon tunggu)")
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')

# Normalisasi: Mengubah rentang pixel 0-255 menjadi 0-1 agar NN belajar lebih cepat
X = X / 255.0

print(f"Data Loaded. Shape: {X.shape}")

Visualisasi Data Tulisan Tangan

In [ ]:
# Mari kita lihat apa yang dilihat oleh AI
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(X[i].reshape(28, 28), cmap='gray')
    plt.title(f"Label: {y[i]}")
    plt.axis('off')
plt.show()

# Log contoh gambar ke W&B
wandb.log({"mnist_examples": [wandb.Image(X[i].reshape(28, 28), caption=f"Label: {y[i]}") for i in range(5)]})

Pelatihan Multi-Layer Perceptron

In [ ]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membangun Arsitektur Jaringan Saraf
# hidden_layer_sizes=(128, 64) artinya ada 2 hidden layer dengan 128 dan 64 neuron
mlp = MLPClassifier(hidden_layer_sizes=(128, 64),
                    activation='relu',
                    solver='adam',
                    learning_rate_init=0.001, # Kecepatan belajar awal
                    alpha=0.0001, # Tambahkan ini: L2 Penalty (Regularisasi)
                    max_iter=20, # Kita batasi iterasi agar cepat untuk latihan
                    verbose=True,
                    random_state=42)

print("Melatih Jaringan Saraf...")
mlp.fit(X_train, y_train)

Evaluasi & Tracking W&B

In [ ]:
y_pred = mlp.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Log kurva kehilangan (Loss Curve) yang menunjukkan proses belajar
for i, loss in enumerate(mlp.loss_curve_):
    wandb.log({"loss": loss, "epoch": i})

wandb.log({"Final_Accuracy": acc})

print(f"\nAkurasi MLP: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Analisis Kesalahan (Confusion Matrix)

In [ ]:
import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Tebakan Model')
plt.ylabel('Digit Asli')
plt.title('Confusion Matrix Pengenalan Angka')
plt.show()

wandb.finish()